# Continual Trajectory-Tracking Policy Learning via Variational Neural Dynamics

This notebook implements the complete two-stage loop from Algorithm 1 of *Continual Robot Policy Learning via Variational Neural Dynamics* for state-based quadrotor figure-eight tracking:

1. collect wind-disturbed tracking rollouts with the latest policy and live encoder;
2. append non-overlapping latent groups to an accumulated replay buffer and jointly update the encoder, residual model, and auxiliary decoder;
3. freeze those models, sample one standard-normal latent per simulated environment, hold it fixed for a complete rollout, and update the latent-conditioned tracking policy by BPTT through `f_prior + D_psi`;
4. repeat.

The CSV latent paired with each wind is never used. Wind labels are not inputs to either network.


In [1]:
import importlib
import os
import time
from datetime import datetime
import numpy as np
import yaml

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
import jax.numpy as jnp
import optax
from flax.core import freeze, unfreeze
from flax.training.train_state import TrainState
from jax.scipy.spatial.transform import Rotation
from matplotlib import pyplot as plt
from orbax.checkpoint import PyTreeCheckpointer

try:
    import wandb
except ModuleNotFoundError:
    wandb = None

from lotf import LOTF_PATH
from lotf.envs import TrajTrackingStateEnv
from lotf.envs.wrappers import MinMaxObservationWrapper
from lotf.modules import (
    LatentConditionedResidualDynamics,
    MLP,
    RecurrentTrajectoryEncoder,
)
from lotf.objects import Quadrotor, RefTrajNames, TrajColumns
import lotf.objects.group_replay_buffer as group_replay_buffer_module

# Reload local buffer edits when this notebook is rerun in a live kernel.
group_replay_buffer_module = importlib.reload(
    group_replay_buffer_module
)
GroupReplayBuffer = group_replay_buffer_module.GroupReplayBuffer
from lotf.utils.pytrees import tree_select

%matplotlib inline



## 1. Reproducibility and continual-learning configuration

The continual-loop, shared-context dataset, dynamics-optimization, and policy-optimization parameters are loaded from the repository-level `config/traj_tracking.yaml`. The paper reports 512 parallel environments and 125 steps (2.5 s at 50 Hz) for trajectory-tracking policy BPTT. `policy_updates_per_round` remains independently configurable.

The tracking reward follows supplement Eq. (6): reference-relative position, velocity, body-rate, acceleration, and attitude Huber costs, plus action smoothness and collision costs. The paper does not publish a task-specific numerical discount, policy learning rate, Huber delta, or collision magnitude, so those tunable quantities remain explicit in the YAML file. `policy_discount=1` and `reward_huber_delta=1` retain the repository's existing undiscounted BPTT and smooth-L1 convention.

Live monitoring uses Weights & Biases. Before running this section for the first time, install it in the notebook kernel with `%pip install wandb`, restart the kernel, and run `wandb login`. Set `wandb_mode = "offline"` below when a network connection is unavailable.

The supplied tracking-policy checkpoint consumes the environment's existing 9-value rotation-matrix observation. The VND context separately uses the paper's `(p, q, v, a)` state-action representation.


In [8]:
seed = 10
master_key = jax.random.key(seed)


def experiment_key(stream_id):
    return jax.random.fold_in(master_key, stream_id)


configuration_path = os.path.abspath(os.path.join(
    LOTF_PATH,
    "..",
    "config",
    "traj_tracking.yaml",
))
with open(configuration_path, "r", encoding="utf-8") as stream:
    experiment_configuration = yaml.safe_load(stream)


def validated_config_section(section_name, expected_keys):
    section = experiment_configuration.get(section_name)
    if not isinstance(section, dict):
        raise TypeError(
            f"Configuration section '{section_name}' must be a mapping"
        )
    expected_keys = set(expected_keys)
    missing_keys = expected_keys - set(section)
    unknown_keys = set(section) - expected_keys
    if missing_keys or unknown_keys:
        raise ValueError(
            f"Invalid keys in configuration section '{section_name}': "
            f"missing={sorted(missing_keys)}, "
            f"unknown={sorted(unknown_keys)}"
        )
    return section


continual_config = validated_config_section(
    "continual_loop",
    {
        "num_continual_rounds",
        "rollouts_per_collection_round",
        "max_collection_attempts_per_round",
        "dynamics_updates_per_round",
        "checkpoint_every_rounds",
        "initialize_vnd_from_pretrained_checkpoint",
        "pretrained_vnd_checkpoint_name",
        "update_normalization_from_continual_buffer",
    },
)
dataset_config = validated_config_section(
    "shared_context_dataset",
    {
        "context_length",
        "dynamics_horizon",
        "latent_dim",
        "state_action_dim",
        "residual_dim",
        "replay_batch_size",
        "prior_evaluation_batch_size",
    },
)
dynamics_config = validated_config_section(
    "dynamics_optimization",
    {
        "dynamics_learning_rate",
        "lambda_rec",
        "lambda_mmd_max",
        "mmd_kernel_sigma",
        "mmd_warmup_updates",
        "mmd_ramp_updates",
        "residual_huber_delta",
    },
)
policy_config = validated_config_section(
    "policy_optimization",
    {
        "policy_num_envs",
        "policy_updates_per_round",
        "policy_rollout_horizon",
        "policy_learning_rate",
        "policy_discount",
        "reward_huber_delta",
    },
)

# Continual loop.
num_continual_rounds = int(continual_config["num_continual_rounds"])
rollouts_per_collection_round = int(
    continual_config["rollouts_per_collection_round"]
)
max_collection_attempts_per_round = int(
    continual_config["max_collection_attempts_per_round"]
)
dynamics_updates_per_round = int(
    continual_config["dynamics_updates_per_round"]
)
checkpoint_every_rounds = int(
    continual_config["checkpoint_every_rounds"]
)
initialize_vnd_from_pretrained_checkpoint = continual_config[
    "initialize_vnd_from_pretrained_checkpoint"
]
pretrained_vnd_checkpoint_name = continual_config[
    "pretrained_vnd_checkpoint_name"
]
update_normalization_from_continual_buffer = continual_config[
    "update_normalization_from_continual_buffer"
]

# Shared-context VND dataset.
context_length = int(dataset_config["context_length"])
dynamics_horizon = int(dataset_config["dynamics_horizon"])
latent_dim = int(dataset_config["latent_dim"])
state_action_dim = int(dataset_config["state_action_dim"])
residual_dim = int(dataset_config["residual_dim"])
replay_batch_size = int(dataset_config["replay_batch_size"])
prior_evaluation_batch_size = int(
    dataset_config["prior_evaluation_batch_size"]
)

# Joint encoder/residual optimization.
dynamics_learning_rate = float(
    dynamics_config["dynamics_learning_rate"]
)
lambda_rec = float(dynamics_config["lambda_rec"])
lambda_mmd_max = float(dynamics_config["lambda_mmd_max"])
mmd_kernel_sigma = float(dynamics_config["mmd_kernel_sigma"])
mmd_warmup_updates = int(dynamics_config["mmd_warmup_updates"])
mmd_ramp_updates = int(dynamics_config["mmd_ramp_updates"])
residual_huber_delta = float(
    dynamics_config["residual_huber_delta"]
)

# Paper-style differentiable policy improvement.
policy_num_envs = int(policy_config["policy_num_envs"])
policy_updates_per_round = int(
    policy_config["policy_updates_per_round"]
)
policy_rollout_horizon = int(
    policy_config["policy_rollout_horizon"]
)
policy_learning_rate = float(policy_config["policy_learning_rate"])
policy_discount = float(policy_config["policy_discount"])
reward_huber_delta = float(policy_config["reward_huber_delta"])

print(f"Loaded training configuration from {configuration_path}")

# Simulation and collection.
sim_dt = 0.02
control_delay = 0.04
max_sim_time = policy_rollout_horizon * sim_dt
reference_trajectory = RefTrajNames.FIG8
collection_thrust_excitation_fraction = 0.03
collection_body_rate_excitation_bound = jnp.array(
    [0.12, 0.12, 0.12], dtype=jnp.float32
)

assert context_length == 20
assert dynamics_horizon > 0
# The final dynamics group may have fewer than dynamics_horizon targets;
# its static padding is excluded by the replay mask.
assert context_length < policy_rollout_horizon
assert policy_rollout_horizon == int(round(max_sim_time / sim_dt))
assert mmd_kernel_sigma == 2.0
assert policy_num_envs > 0 and policy_updates_per_round > 0

# Live experiment monitoring. Set enable_wandb=False to disable it.
enable_wandb = True
wandb_project = "continual-vnd-trajectory-tracking"
wandb_entity = None
wandb_run_name = None
wandb_mode = "online"  # Use "offline" when disconnected.
wandb_dynamics_log_every = 25
wandb_histogram_every_rounds = 10
assert wandb_dynamics_log_every > 0

if enable_wandb and wandb is None:
    raise ModuleNotFoundError(
        "Weights & Biases is not installed in this kernel. Run "
        "`%pip install wandb`, restart the kernel, then run "
        "`wandb login`; or set enable_wandb=False."
    )

wandb_run = None
if enable_wandb:
    wandb_run = wandb.init(
        project=wandb_project,
        entity=wandb_entity,
        name=wandb_run_name,
        mode=wandb_mode,
        config={
            "seed": seed,
            "num_continual_rounds": num_continual_rounds,
            "rollouts_per_collection_round": rollouts_per_collection_round,
            "dynamics_updates_per_round": dynamics_updates_per_round,
            "context_length": context_length,
            "dynamics_horizon": dynamics_horizon,
            "latent_dim": latent_dim,
            "replay_batch_size": replay_batch_size,
            "dynamics_learning_rate": dynamics_learning_rate,
            "lambda_rec": lambda_rec,
            "lambda_mmd_max": lambda_mmd_max,
            "mmd_kernel_sigma": mmd_kernel_sigma,
            "mmd_warmup_updates": mmd_warmup_updates,
            "mmd_ramp_updates": mmd_ramp_updates,
            "residual_huber_delta": residual_huber_delta,
            "residual_dim": residual_dim,
            "reference_trajectory": reference_trajectory.value,
            "policy_num_envs": policy_num_envs,
            "policy_updates_per_round": policy_updates_per_round,
            "policy_rollout_horizon": policy_rollout_horizon,
            "policy_learning_rate": policy_learning_rate,
            "policy_discount": policy_discount,
            "reward_huber_delta": reward_huber_delta,
            "sim_dt": sim_dt,
            "control_delay": control_delay,
        },
    )
    wandb_run.define_metric("dynamics/update")
    wandb_run.define_metric(
        "dynamics/*", step_metric="dynamics/update"
    )
    wandb_run.define_metric("continual/round")
    for metric_prefix in (
        "policy", "dynamics_round", "replay", "collection",
        "encoder", "parameters", "timing",
    ):
        wandb_run.define_metric(
            f"{metric_prefix}/*", step_metric="continual/round"
        )
    print(f"W&B run: {wandb_run.url or wandb_run.id}")



Loaded training configuration from /home/dimitria/PhD_codes/learning_on_the_fly/config/traj_tracking.yaml


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


W&B run: https://wandb.ai/dimitriasilveria-ds-queen-s-university/continual-vnd-trajectory-tracking/runs/pw8nbv97


## 2. Nominal and disturbed trajectory-tracking environments

Both environments track the same figure-eight reference with randomized initial states near the current waypoint and retain the original 0.04 s action delay. The deployment environment applies the true episode-level wind acceleration. The differentiable policy environment is nominal; its successor is corrected explicitly by the frozen learned residual.


In [9]:
nominal_dynamics_configuration = {
    "use_high_fidelity": False,
    "use_forward_residual": False,
}


def make_tracking_environment(apply_hidden_acceleration):
    environment = TrajTrackingStateEnv(
        max_steps_in_episode=policy_rollout_horizon,
        dt=sim_dt,
        delay=control_delay,
        quad_obj=Quadrotor.from_name(
            "example_quad", nominal_dynamics_configuration
        ),
        ref_traj_name=reference_trajectory,
        skip_start=True,
        apply_hidden_acceleration=apply_hidden_acceleration,
    )
    return MinMaxObservationWrapper(environment)


deployment_env = make_tracking_environment(
    apply_hidden_acceleration=True
)
policy_sim_env = make_tracking_environment(
    apply_hidden_acceleration=False
)

action_dim = deployment_env.action_space.shape[0]
obs_dim = deployment_env.observation_space.shape[0]
dummy_residual_params = {}

collection_action_excitation_bound = jnp.concatenate([
    (
        collection_thrust_excitation_fraction
        * deployment_env.hovering_action[0]
    )[None],
    collection_body_rate_excitation_bound,
])

print("Trajectory-tracking environment configuration")
print(f"  reference trajectory: {reference_trajectory.value}")
print(f"  observation dimension: {obs_dim}")
print(f"  action dimension:      {action_dim}")
print(f"  control delay:         {control_delay:.3f} s")
print(f"  rollout duration:      {max_sim_time:.2f} s")
print(f"  action excitation:     {collection_action_excitation_bound}")


Trajectory-tracking environment configuration
  reference trajectory: fig8
  observation dimension: 27
  action dimension:      4
  control delay:         0.040 s
  rollout duration:      2.50 s
  action excitation:     [0.0565056 0.12      0.12      0.12     ]


## 3. Load the base policy and add the latent input without changing its initial behavior

The loaded policy has observation-only input. The continual policy has input `[normalized task observation, z]`. Its observation rows and all later layers are copied exactly from the checkpoint, while the new latent rows of the first kernel are initialized to zero. Consequently, the first deployment round uses exactly the previously trained policy for every latent.



In [10]:
policy_name = "traj_tracking_params"
policy_checkpoint_path = os.path.abspath(os.path.join(
    LOTF_PATH, "..", "checkpoints", "policy", policy_name
))
checkpointer = PyTreeCheckpointer()
base_policy_params = checkpointer.restore(policy_checkpoint_path)

base_policy_net = MLP(
    [obs_dim, 512, 512, action_dim],
    initial_scale=0.01,
    action_bias=deployment_env.hovering_action,
)
latent_policy_net = MLP(
    [obs_dim + latent_dim, 512, 512, action_dim],
    initial_scale=0.01,
    action_bias=deployment_env.hovering_action,
)

expanded_policy_params = unfreeze(
    latent_policy_net.initialize(experiment_key(10))
)
loaded_policy_params = unfreeze(base_policy_params)

assert tuple(expanded_policy_params["params"]) == tuple(
    loaded_policy_params["params"]
)
for layer_name in loaded_policy_params["params"]:
    if layer_name == "Dense_0":
        old_kernel = loaded_policy_params["params"][layer_name]["kernel"]
        new_kernel = expanded_policy_params["params"][layer_name]["kernel"]
        assert old_kernel.shape == (obs_dim, 512)
        assert new_kernel.shape == (obs_dim + latent_dim, 512)
        expanded_policy_params["params"][layer_name]["kernel"] = (
            new_kernel.at[:obs_dim].set(old_kernel).at[obs_dim:].set(0.0)
        )
        expanded_policy_params["params"][layer_name]["bias"] = (
            loaded_policy_params["params"][layer_name]["bias"]
        )
    else:
        expanded_policy_params["params"][layer_name] = (
            loaded_policy_params["params"][layer_name]
        )

expanded_policy_params = freeze(expanded_policy_params)
policy_train_state = TrainState.create(
    apply_fn=latent_policy_net.apply,
    params=expanded_policy_params,
    tx=optax.adam(policy_learning_rate),
)

policy_equivalence_obs = jnp.zeros((4, obs_dim), dtype=jnp.float32)
policy_equivalence_z = jax.random.normal(
    experiment_key(11), (4, latent_dim)
)
base_actions = base_policy_net.apply(
    base_policy_params, policy_equivalence_obs
)
expanded_actions = latent_policy_net.apply(
    policy_train_state.params,
    jnp.concatenate([policy_equivalence_obs, policy_equivalence_z], axis=-1),
)
assert jnp.allclose(base_actions, expanded_actions, atol=1e-6)
print(f"Loaded base policy from {policy_checkpoint_path}")
print("Latent-conditioned policy initially reproduces it exactly.")



Loaded base policy from /home/dimitria/PhD_codes/learning_on_the_fly/checkpoints/policy/traj_tracking_params
Latent-conditioned policy initially reproduces it exactly.


/home/dimitria/venvs/diff-sim-env/lib/python3.12/site-packages/orbax/checkpoint/type_handlers.py:1330: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


## 4. Create the architecture and restore the pretrained tracking VND model

The GRU encoder uses a 128-D projection and hidden state. The shared residual backbone has 256 hidden units, two FiLM-conditioned blocks, and separate position, velocity, and local orientation heads. One latent is broadcast across all valid transitions assigned to its context. By default, the encoder, residual model, auxiliary decoder, and target-normalization statistics are restored together from `traj_tracking_complete_mmd`; Adam starts with a fresh optimizer state.


In [11]:
trajectory_encoder = RecurrentTrajectoryEncoder(
    state_action_dim=state_action_dim,
    context_length=context_length,
    latent_dim=latent_dim,
    projection_dim=128,
    hidden_dim=128,
)
residual_model = LatentConditionedResidualDynamics(
    state_action_dim=state_action_dim,
    latent_dim=latent_dim,
    hidden_dim=256,
    num_blocks=2,
    predict_orientation=True,
)
context_decoder = MLP(
    [latent_dim, 256, 256, context_length * state_action_dim],
    nonlinearity=jax.nn.gelu,
)

key_encoder, key_residual, key_decoder = jax.random.split(
    experiment_key(20), 3
)
random_initial_joint_params = {
    "encoder": trajectory_encoder.initialize(key_encoder),
    "residual": residual_model.initialize(key_residual),
    "decoder": context_decoder.initialize(key_decoder),
}

pretrained_vnd_checkpoint = None
pretrained_vnd_completed_updates = 0
if initialize_vnd_from_pretrained_checkpoint:
    pretrained_vnd_checkpoint_path = os.path.abspath(os.path.join(
        LOTF_PATH,
        "..",
        "checkpoints",
        "residual_dynamics",
        pretrained_vnd_checkpoint_name,
    ))
    pretrained_vnd_checkpoint = PyTreeCheckpointer().restore(
        pretrained_vnd_checkpoint_path
    )
    pretrained_configuration = pretrained_vnd_checkpoint["configuration"]
    expected_configuration = {
        "context_length": context_length,
        "dynamics_horizon": dynamics_horizon,
        "state_action_dim": state_action_dim,
        "latent_dim": latent_dim,
        "residual_dim": residual_dim,
    }
    for name, expected_value in expected_configuration.items():
        restored_value = int(jax.device_get(
            pretrained_configuration[name]
        ))
        if restored_value != expected_value:
            raise ValueError(
                f"Pretrained {name}={restored_value}, but the continual "
                f"notebook requires {expected_value}"
            )
    if set(pretrained_vnd_checkpoint["params"]) != {
        "encoder", "residual", "decoder"
    }:
        raise ValueError(
            "Pretrained checkpoint must contain encoder, residual, "
            "and decoder parameter groups"
        )
    initial_joint_params = pretrained_vnd_checkpoint["params"]
    startup_input_mean = pretrained_vnd_checkpoint["input_mean"]
    startup_input_std = pretrained_vnd_checkpoint["input_std"]
    startup_residual_target_mean = (
        pretrained_vnd_checkpoint["residual_target_mean"]
    )
    startup_residual_target_std = (
        pretrained_vnd_checkpoint["residual_target_std"]
    )
    if startup_residual_target_mean.shape != (residual_dim,):
        raise ValueError(
            "Pretrained residual statistics do not match the "
            f"configured {residual_dim}-D tracking residual"
        )
    pretrained_vnd_completed_updates = int(jax.device_get(
        pretrained_configuration.get(
            "num_training_steps", jnp.asarray(0)
        )
    ))
    print("Loaded pretrained encoder, residual, decoder, and statistics:")
    print(f"  {pretrained_vnd_checkpoint_path}")
    print(
        f"  continuing MMD schedule from dynamics update "
        f"{pretrained_vnd_completed_updates:,}"
    )
else:
    initial_joint_params = random_initial_joint_params
    startup_input_mean = jnp.zeros(
        (state_action_dim,), dtype=jnp.float32
    )
    startup_input_std = jnp.ones(
        (state_action_dim,), dtype=jnp.float32
    )
    startup_residual_target_mean = jnp.zeros(
        (residual_dim,), dtype=jnp.float32
    )
    startup_residual_target_std = jnp.ones(
        (residual_dim,), dtype=jnp.float32
    )


def joint_forward(params, contexts, normalized_dynamics_inputs):
    latents = trajectory_encoder.apply(params["encoder"], contexts)
    expanded_latents = jnp.broadcast_to(
        latents[..., None, :],
        normalized_dynamics_inputs.shape[:-1] + (latent_dim,),
    )
    normalized_residuals = residual_model.apply(
        params["residual"],
        normalized_dynamics_inputs,
        expanded_latents,
    )
    reconstructed_contexts = context_decoder.apply(
        params["decoder"], latents
    ).reshape(contexts.shape)
    return latents, normalized_residuals, reconstructed_contexts


joint_train_state = TrainState.create(
    apply_fn=joint_forward,
    params=initial_joint_params,
    tx=optax.adam(dynamics_learning_rate),
)



/home/dimitria/venvs/diff-sim-env/lib/python3.12/site-packages/orbax/checkpoint/type_handlers.py:1330: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Loaded pretrained encoder, residual, decoder, and statistics:
  /home/dimitria/PhD_codes/learning_on_the_fly/checkpoints/residual_dynamics/traj_tracking_complete_mmd_best
  continuing MMD schedule from dynamics update 80,000


## 5. State representation, normalization, and accumulated replay buffer

The replay buffer stores raw contexts, raw dynamics inputs, and physical residual targets. Every sampled old context is re-encoded with the latest encoder parameters. Because the pretrained checkpoint does not contain the original replay samples or sufficient statistics, its matched input/target statistics remain fixed by default; set `update_normalization_from_continual_buffer=True` only if deliberately replacing them with statistics from the new buffer. Position and velocity inputs are clipped to `[-5, 5]` only after normalization.



In [12]:
def state_vector_from_env_state(env_state):
    quaternion_xyzw = Rotation.from_matrix(
        env_state.quadrotor_state.R
    ).as_quat()
    quaternion_wxyz = jnp.concatenate(
        [quaternion_xyzw[..., 3:], quaternion_xyzw[..., :3]], axis=-1
    )
    return jnp.concatenate([
        env_state.quadrotor_state.p,
        quaternion_wxyz,
        env_state.quadrotor_state.v,
    ], axis=-1)


def orientation_residual(observed_rotation, nominal_rotation):
    relative_rotation = (
        jnp.swapaxes(nominal_rotation, -1, -2) @ observed_rotation
    )
    return Rotation.from_matrix(relative_rotation).as_rotvec()


def rotation_matrix_from_rotvec(rotation_vector):
    # Rodrigues' formula with zero-angle-safe coefficients. JAX's
    # generic Rotation.from_rotvec has an undefined 0/0 derivative at
    # exactly zero, which can poison BPTT when the residual is zero.
    x, y, z = jnp.moveaxis(rotation_vector, -1, 0)
    zeros = jnp.zeros_like(x)
    skew = jnp.stack([
        zeros, -z, y,
        z, zeros, -x,
        -y, x, zeros,
    ], axis=-1).reshape(rotation_vector.shape[:-1] + (3, 3))
    angle_squared = jnp.sum(jnp.square(rotation_vector), axis=-1)
    safe_angle = jnp.sqrt(jnp.maximum(
        angle_squared, jnp.finfo(rotation_vector.dtype).eps
    ))
    coefficient_a_regular = jnp.sin(safe_angle) / safe_angle
    half_angle = 0.5 * safe_angle
    coefficient_b_regular = 0.5 * jnp.square(
        jnp.sin(half_angle) / half_angle
    )
    coefficient_a_series = (
        1.0 - angle_squared / 6.0 + angle_squared**2 / 120.0
    )
    coefficient_b_series = (
        0.5 - angle_squared / 24.0 + angle_squared**2 / 720.0
    )
    use_series = angle_squared < 1e-4
    coefficient_a = jnp.where(
        use_series, coefficient_a_series, coefficient_a_regular
    )
    coefficient_b = jnp.where(
        use_series, coefficient_b_series, coefficient_b_regular
    )
    identity = jnp.broadcast_to(
        jnp.eye(3, dtype=rotation_vector.dtype), skew.shape
    )
    return (
        identity
        + coefficient_a[..., None, None] * skew
        + coefficient_b[..., None, None] * (skew @ skew)
    )


def compose_orientation_residual(nominal_rotation, rotation_vector):
    correction_rotation = rotation_matrix_from_rotvec(rotation_vector)
    return nominal_rotation @ correction_rotation


def rotation_matrix_from_wxyz(quaternion_wxyz):
    quaternion_xyzw = jnp.concatenate([
        quaternion_wxyz[..., 1:], quaternion_wxyz[..., :1]
    ], axis=-1)
    return Rotation.from_quat(quaternion_xyzw).as_matrix()


def normalize_state_actions(raw_state_actions, input_mean, input_std):
    normalized = (raw_state_actions - input_mean) / input_std
    normalized = normalized.at[..., 0:3].set(
        jnp.clip(normalized[..., 0:3], -5.0, 5.0)
    )
    normalized = normalized.at[..., 7:10].set(
        jnp.clip(normalized[..., 7:10], -5.0, 5.0)
    )
    return normalized


replay_buffer = GroupReplayBuffer(
    context_length=context_length,
    dynamics_horizon=dynamics_horizon,
    state_action_dim=state_action_dim,
    residual_dim=residual_dim,
)



## 6. Stage 1A — collect rollouts with the latest policy and live encoder

Before the first complete context exists, the latent is zero. Thereafter the encoder consumes the rolling 20-step normalized state-action window. Small bounded action excitation is applied only during deployment-data collection. The true wind stays constant for each rollout, while the CSV's preassigned latent is ignored.



In [13]:
@jax.jit
def collect_deployment_rollouts(
    policy_params,
    joint_params,
    input_mean,
    input_std,
    collection_key,
):
    key_reset, key_scan = jax.random.split(collection_key)
    reset_keys = jax.random.split(
        key_reset, rollouts_per_collection_round
    )
    initial_states, initial_observations = jax.vmap(
        lambda reset_key: deployment_env.reset(reset_key)
    )(reset_keys)
    initial_contexts = jnp.zeros(
        (
            rollouts_per_collection_round,
            context_length,
            state_action_dim,
        ),
        dtype=jnp.float32,
    )

    def collection_step(carry, step_key):
        env_states, observations, contexts, context_count = carry
        key_excitation, key_environment = jax.random.split(step_key)

        encoded_latents = trajectory_encoder.apply(
            joint_params["encoder"], contexts
        )
        policy_latents = jnp.where(
            context_count >= context_length,
            encoded_latents,
            jnp.zeros_like(encoded_latents),
        )
        policy_inputs = jnp.concatenate(
            [observations, policy_latents], axis=-1
        )
        nominal_actions = latent_policy_net.apply(
            policy_params, policy_inputs
        )
        excitation = jax.random.uniform(
            key_excitation,
            shape=nominal_actions.shape,
            minval=-collection_action_excitation_bound,
            maxval=collection_action_excitation_bound,
        )
        actions = jnp.clip(
            nominal_actions + excitation,
            deployment_env.action_space.low,
            deployment_env.action_space.high,
        )

        step_keys = jax.random.split(
            key_environment, rollouts_per_collection_round
        )
        transitions = jax.vmap(
            lambda state, action, key: deployment_env._step(
                state, action, dummy_residual_params, key
            )
        )(env_states, actions, step_keys)

        raw_state_actions = jnp.concatenate(
            [state_vector_from_env_state(env_states), actions], axis=-1
        )
        normalized_state_actions = normalize_state_actions(
            raw_state_actions, input_mean, input_std
        )
        next_contexts = jnp.roll(contexts, shift=-1, axis=1)
        next_contexts = next_contexts.at[:, -1, :].set(
            normalized_state_actions
        )
        done = jnp.logical_or(
            transitions.terminated, transitions.truncated
        )
        next_carry = (
            transitions.state,
            transitions.obs,
            next_contexts,
            context_count + 1,
        )
        outputs = (transitions.state, actions, done)
        return next_carry, outputs

    scan_keys = jax.random.split(key_scan, policy_rollout_horizon)
    _, (successor_states, actions, done) = jax.lax.scan(
        collection_step,
        (
            initial_states,
            initial_observations,
            initial_contexts,
            jnp.asarray(0, dtype=jnp.int32),
        ),
        scan_keys,
    )
    time_major_states = jax.tree.map(
        lambda initial, successors: jnp.concatenate(
            [initial[None], successors], axis=0
        ),
        initial_states,
        successor_states,
    )
    rollout_states = jax.tree.map(
        lambda leaf: jnp.swapaxes(leaf, 0, 1), time_major_states
    )
    return rollout_states, jnp.swapaxes(actions, 0, 1), jnp.swapaxes(done, 0, 1)



## 7. Stage 1B — construct non-overlapping groups and append them to the buffer

For each rollout, a valid context encodes one latent for up to `dynamics_horizon` following target transitions. Group starts advance by exactly `dynamics_horizon`, so no target transition is assigned to two latent codes. A terminal group is retained whenever its context and at least one following transition are valid; unavailable targets are padded only for static shapes and excluded by a replayed boolean mask. Nominal successors use the same initial simulator state, recorded action, delay buffer, and `_step` implementation as collection.



In [14]:
@jax.jit
def evaluate_nominal_prior_batch(states, actions, keys):
    return jax.vmap(
        lambda state, action, key: policy_sim_env._env._step(
            state, action, dummy_residual_params, key
        ).state.quadrotor_state
    )(states, actions, keys)


def build_groups_from_collection(
    rollout_states, rollout_actions, rollout_done, prior_key
):
    num_rollouts = rollout_actions.shape[0]
    num_steps = rollout_actions.shape[1]
    current_states = jax.tree.map(
        lambda leaf: leaf[:, :-1], rollout_states
    )
    observed_successors = jax.tree.map(
        lambda leaf: leaf[:, 1:], rollout_states.quadrotor_state
    )
    raw_state_actions = jnp.concatenate([
        state_vector_from_env_state(current_states), rollout_actions
    ], axis=-1)
    assert raw_state_actions.shape == (
        num_rollouts, num_steps, state_action_dim
    )

    flat_current_states = jax.tree.map(
        lambda leaf: leaf.reshape((-1,) + leaf.shape[2:]), current_states
    )
    flat_actions = rollout_actions.reshape(-1, action_dim)
    num_transitions = flat_actions.shape[0]
    flat_prior_keys = jax.random.split(prior_key, num_transitions)
    num_padded = (
        (num_transitions + prior_evaluation_batch_size - 1)
        // prior_evaluation_batch_size
        * prior_evaluation_batch_size
    )
    padding_indices = jnp.minimum(
        jnp.arange(num_padded), num_transitions - 1
    )
    padded_states = jax.tree.map(
        lambda leaf: leaf[padding_indices], flat_current_states
    )
    padded_actions = flat_actions[padding_indices]
    padded_keys = flat_prior_keys[padding_indices]

    nominal_chunks = []
    for start in range(0, num_padded, prior_evaluation_batch_size):
        stop = start + prior_evaluation_batch_size
        nominal_chunks.append(evaluate_nominal_prior_batch(
            jax.tree.map(lambda leaf: leaf[start:stop], padded_states),
            padded_actions[start:stop],
            padded_keys[start:stop],
        ))
    nominal_successors = jax.tree.map(
        lambda *chunks: jnp.concatenate(chunks, axis=0)[:num_transitions],
        *nominal_chunks,
    )
    nominal_successors = jax.tree.map(
        lambda leaf: leaf.reshape(
            (num_rollouts, num_steps) + leaf.shape[1:]
        ),
        nominal_successors,
    )
    residual_targets = jnp.concatenate([
        observed_successors.p - nominal_successors.p,
        observed_successors.v - nominal_successors.v,
        orientation_residual(
            observed_successors.R, nominal_successors.R
        ),
    ], axis=-1)
    assert residual_targets.shape[-1] == residual_dim

    valid_transition = (
        jnp.cumsum(rollout_done, axis=1) - rollout_done
    ) == 0
    num_candidate_groups = (
        num_steps - context_length + dynamics_horizon - 1
    ) // dynamics_horizon
    if num_candidate_groups <= 0:
        raise ValueError(
            "Need at least context_length + 1 rollout steps"
        )
    group_starts = jnp.arange(num_candidate_groups) * dynamics_horizon
    context_indices = (
        group_starts[:, None] + jnp.arange(context_length)[None, :]
    )
    unbounded_target_indices = (
        group_starts[:, None]
        + context_length
        + jnp.arange(dynamics_horizon)[None, :]
    )
    target_exists = unbounded_target_indices < num_steps
    target_indices = jnp.minimum(unbounded_target_indices, num_steps - 1)
    existing_target_indices = unbounded_target_indices[target_exists]
    assert jnp.unique(existing_target_indices).size == (
        existing_target_indices.size
    )

    grouped_contexts = raw_state_actions[:, context_indices]
    grouped_inputs = raw_state_actions[:, target_indices]
    grouped_targets = residual_targets[:, target_indices]
    context_is_valid = valid_transition[:, context_indices].all(axis=-1)
    grouped_target_masks = (
        valid_transition[:, target_indices]
        & target_exists[None, :, :]
    )
    valid_groups = context_is_valid & grouped_target_masks.any(axis=-1)
    return (
        grouped_contexts[valid_groups],
        grouped_inputs[valid_groups],
        grouped_targets[valid_groups],
        grouped_target_masks[valid_groups],
    )



## 8. Stage 1C — joint VND update with a fixed prior bank

`fixed_prior_bank` is sampled in Stage 2 and reused for every dynamics update in the next Stage 1. The first Stage 1 receives one reproducible standard-normal bank. No standard-normal vectors are sampled inside the dynamics optimizer. Old contexts are sampled from the accumulated raw buffer and re-encoded with current encoder parameters on every visit.



In [15]:
def biased_rbf_mmd(encoded_latents, prior_latents):
    def kernel(left, right):
        left_norm = jnp.sum(jnp.square(left), axis=-1)[:, None]
        right_norm = jnp.sum(jnp.square(right), axis=-1)[None, :]
        squared_distance = jnp.maximum(
            left_norm + right_norm - 2.0 * (left @ right.T), 0.0
        )
        return jnp.exp(
            -squared_distance / (2.0 * mmd_kernel_sigma**2)
        )

    return (
        kernel(encoded_latents, encoded_latents).mean()
        + kernel(prior_latents, prior_latents).mean()
        - 2.0 * kernel(encoded_latents, prior_latents).mean()
    )


def mmd_weight_at_update(update_index):
    ramp_fraction = jnp.clip(
        (update_index - mmd_warmup_updates) / mmd_ramp_updates,
        0.0,
        1.0,
    )
    return lambda_mmd_max * ramp_fraction


def joint_vnd_loss(
    params,
    raw_contexts,
    raw_dynamics_inputs,
    raw_targets,
    raw_target_masks,
    prior_latents,
    input_mean,
    input_std,
    target_mean,
    target_std,
    mmd_weight,
):
    normalized_contexts = normalize_state_actions(
        raw_contexts, input_mean, input_std
    )
    normalized_dynamics_inputs = normalize_state_actions(
        raw_dynamics_inputs, input_mean, input_std
    )
    normalized_targets = (raw_targets - target_mean) / target_std
    latents, normalized_predictions, reconstructed_contexts = joint_forward(
        params, normalized_contexts, normalized_dynamics_inputs
    )
    per_component_dynamics_loss = optax.huber_loss(
        normalized_predictions,
        normalized_targets,
        delta=residual_huber_delta,
    )
    per_transition_dynamics_loss = per_component_dynamics_loss.mean(
        axis=-1
    )
    target_mask_weights = raw_target_masks.astype(
        per_transition_dynamics_loss.dtype
    )
    valid_targets_per_group = jnp.maximum(
        target_mask_weights.sum(axis=-1), 1.0
    )
    dynamics_loss = jnp.mean(
        (per_transition_dynamics_loss * target_mask_weights).sum(axis=-1)
        / valid_targets_per_group
    )
    reconstruction_loss = jnp.sum(
        jnp.square(reconstructed_contexts - normalized_contexts),
        axis=(-2, -1),
    ).mean()
    mmd_loss = biased_rbf_mmd(latents, prior_latents)
    total_loss = (
        dynamics_loss
        + lambda_rec * reconstruction_loss
        + mmd_weight * mmd_loss
    )
    return total_loss, (dynamics_loss, reconstruction_loss, mmd_loss)


@jax.jit
def dynamics_optimizer_step(
    train_state,
    raw_contexts,
    raw_dynamics_inputs,
    raw_targets,
    raw_target_masks,
    prior_latents,
    input_mean,
    input_std,
    target_mean,
    target_std,
    global_update,
):
    mmd_weight = mmd_weight_at_update(global_update)
    (loss, components), gradients = jax.value_and_grad(
        joint_vnd_loss, has_aux=True
    )(
        train_state.params,
        raw_contexts,
        raw_dynamics_inputs,
        raw_targets,
        raw_target_masks,
        prior_latents,
        input_mean,
        input_std,
        target_mean,
        target_std,
        mmd_weight,
    )
    gradient_norm = optax.global_norm(gradients)
    encoder_gradient_norm = optax.global_norm(gradients["encoder"])
    residual_gradient_norm = optax.global_norm(gradients["residual"])
    decoder_gradient_norm = optax.global_norm(gradients["decoder"])
    previous_params = train_state.params
    train_state = train_state.apply_gradients(grads=gradients)
    parameter_updates = jax.tree.map(
        lambda updated, previous: updated - previous,
        train_state.params,
        previous_params,
    )
    update_norm = optax.global_norm(parameter_updates)
    encoder_update_norm = optax.global_norm(parameter_updates["encoder"])
    residual_update_norm = optax.global_norm(parameter_updates["residual"])
    decoder_update_norm = optax.global_norm(parameter_updates["decoder"])
    return train_state, jnp.stack([
        loss, components[0], components[1], components[2], mmd_weight,
        gradient_norm, update_norm,
        encoder_gradient_norm, residual_gradient_norm,
        decoder_gradient_norm, encoder_update_norm,
        residual_update_norm, decoder_update_norm,
    ])


def run_dynamics_stage(
    train_state,
    replay,
    fixed_prior_bank,
    input_mean,
    input_std,
    target_mean,
    target_std,
    first_global_update,
    stage_key,
):
    metrics = []
    for local_update in range(dynamics_updates_per_round):
        global_update = first_global_update + local_update
        batch_key = jax.random.fold_in(stage_key, local_update)
        batch = replay.sample(batch_key, replay_batch_size)
        prior_indices = (
            jnp.arange(replay_batch_size)
            + global_update * replay_batch_size
        ) % fixed_prior_bank.shape[0]
        prior_latents = fixed_prior_bank[prior_indices]
        train_state, update_metrics = dynamics_optimizer_step(
            train_state,
            *batch,
            prior_latents,
            input_mean,
            input_std,
            target_mean,
            target_std,
            jnp.asarray(global_update),
        )
        metrics.append(update_metrics)
        should_log = (
            local_update % wandb_dynamics_log_every == 0
            or local_update == dynamics_updates_per_round - 1
        )
        if wandb_run is not None and should_log:
            host_update_metrics = np.asarray(
                jax.device_get(update_metrics)
            )
            wandb_run.log({
                "dynamics/update": global_update + 1,
                "dynamics/total_loss": host_update_metrics[0],
                "dynamics/L_dyn": host_update_metrics[1],
                "dynamics/L_rec": host_update_metrics[2],
                "dynamics/weighted_L_rec": (
                    lambda_rec * host_update_metrics[2]
                ),
                "dynamics/L_mmd": host_update_metrics[3],
                "dynamics/lambda_mmd": host_update_metrics[4],
                "dynamics/weighted_L_mmd": (
                    host_update_metrics[4] * host_update_metrics[3]
                ),
                "dynamics/gradient_norm": host_update_metrics[5],
                "dynamics/update_norm": host_update_metrics[6],
                "dynamics/encoder_gradient_norm": host_update_metrics[7],
                "dynamics/residual_gradient_norm": host_update_metrics[8],
                "dynamics/decoder_gradient_norm": host_update_metrics[9],
                "dynamics/encoder_update_norm": host_update_metrics[10],
                "dynamics/residual_update_norm": host_update_metrics[11],
                "dynamics/decoder_update_norm": host_update_metrics[12],
            })
    metrics = jnp.stack(metrics)
    metrics.block_until_ready()
    return train_state, metrics



## 9. Stage 2 — freeze VND and improve the tracking policy by BPTT

At every policy optimizer update, the notebook samples one independent `z ~ N(0, I)` per parallel environment. That latent is held fixed throughout the complete 2.5 s recursive rollout and conditions both the policy and residual model. Gradients are taken only with respect to policy parameters while still propagating through the frozen residual model's state/action Jacobians.

The reward implements supplement Eq. (6): Huber penalties on position, velocity, body-rate, acceleration, and attitude errors relative to the current figure-eight waypoint, plus `0.15` times the action-change penalty and the existing remaining-horizon collision convention. The weights are `1, 0.1, 0.01, 0.01, 1, 0.15`, respectively.


In [16]:
def huber_magnitude(error, delta=reward_huber_delta):
    squared_magnitude = jnp.sum(jnp.square(error), axis=-1)
    safe_magnitude = jnp.sqrt(jnp.maximum(
        squared_magnitude, jnp.finfo(error.dtype).eps
    ))
    return jnp.where(
        squared_magnitude <= delta**2,
        0.5 * squared_magnitude,
        delta * (safe_magnitude - 0.5 * delta),
    )


def huber_scalar(error, delta=reward_huber_delta):
    magnitude = jnp.abs(error)
    quadratic = jnp.minimum(magnitude, delta)
    linear = magnitude - quadratic
    return 0.5 * jnp.square(quadratic) + delta * linear


def rotation_huber_cost(relative_rotation, delta=reward_huber_delta):
    cosine = jnp.clip(
        (jnp.trace(relative_rotation, axis1=-2, axis2=-1) - 1.0) / 2.0,
        -1.0,
        1.0,
    )
    one_minus_cosine = jnp.maximum(1.0 - cosine, 0.0)
    angle_squared_near_identity = (
        2.0 * one_minus_cosine
        + jnp.square(one_minus_cosine) / 3.0
        + 4.0 * one_minus_cosine**3 / 45.0
    )
    safe_cosine = jnp.clip(cosine, -1.0 + 1e-7, 1.0 - 1e-7)
    angle_squared_regular = jnp.square(jnp.arccos(safe_cosine))
    angle_squared = jnp.where(
        one_minus_cosine < 1e-4,
        angle_squared_near_identity,
        angle_squared_regular,
    )
    safe_angle = jnp.sqrt(jnp.maximum(
        angle_squared, jnp.finfo(relative_rotation.dtype).eps
    ))
    return jnp.where(
        angle_squared <= delta**2,
        0.5 * angle_squared,
        delta * (safe_angle - 0.5 * delta),
    )


def trajectory_tracking_reward(
    last_states, next_states, commanded_actions, remaining_steps
):
    quad_states = next_states.quadrotor_state
    target_indices = jnp.minimum(
        next_states.init_ref_traj_idx + next_states.step_idx,
        policy_sim_env._env.num_ref_traj_points - 1,
    )
    reference = policy_sim_env._env.ref_traj[target_indices]
    reference_position = reference[:, TrajColumns.POS.slice]
    reference_rotation = rotation_matrix_from_wxyz(
        reference[:, TrajColumns.QUAT.slice]
    )
    reference_velocity = reference[:, TrajColumns.VEL.slice]
    reference_body_rate = reference[:, TrajColumns.OMEGA.slice]
    reference_acceleration = reference[:, TrajColumns.ACC.slice]

    position_error = jnp.linalg.norm(
        quad_states.p - reference_position, axis=-1
    )
    position_cost = huber_magnitude(
        quad_states.p - reference_position
    )
    velocity_cost = huber_magnitude(
        quad_states.v - reference_velocity
    )
    body_rate_cost = huber_magnitude(
        quad_states.omega - reference_body_rate
    )
    acceleration_cost = huber_magnitude(
        quad_states.acc - reference_acceleration
    )
    relative_attitude = (
        jnp.swapaxes(reference_rotation, -1, -2) @ quad_states.R
    )
    attitude_cost = rotation_huber_cost(relative_attitude)
    previous_actions = last_states.last_actions[:, -1, :]
    action_change_cost = huber_magnitude(
        commanded_actions - previous_actions
    )
    base_cost = (
        position_cost
        + 0.1 * velocity_cost
        + 0.01 * body_rate_cost
        + 0.01 * acceleration_cost
        + attitude_cost
        + 0.15 * action_change_cost
    )
    collision = jax.vmap(policy_sim_env._env._is_colliding)(next_states)
    collision_indicator = jax.lax.stop_gradient(
        collision.astype(base_cost.dtype)
    )
    collision_cost = collision_indicator * remaining_steps * base_cost
    reward = -sim_dt * (base_cost + collision_cost)
    return reward, position_error


delay_first_segment = (
    policy_sim_env.delay
    - (policy_sim_env.num_last_actions - 2) * policy_sim_env.dt
)
effective_action_buffer_index = int(delay_first_segment < policy_sim_env.dt)


def corrected_policy_step(
    env_states,
    actions,
    rollout_latents,
    joint_params,
    input_mean,
    input_std,
    target_mean,
    target_std,
    step_keys,
):
    clipped_actions = jnp.clip(
        actions,
        policy_sim_env.action_space.low,
        policy_sim_env.action_space.high,
    )
    nominal_transitions = jax.vmap(
        lambda state, action, key: policy_sim_env._env._step(
            state, action, dummy_residual_params, key
        )
    )(env_states, clipped_actions, step_keys)
    raw_residual_inputs = jnp.concatenate([
        state_vector_from_env_state(env_states), clipped_actions
    ], axis=-1)
    normalized_residual_inputs = normalize_state_actions(
        raw_residual_inputs, input_mean, input_std
    )
    normalized_corrections = residual_model.apply(
        joint_params["residual"],
        normalized_residual_inputs,
        rollout_latents,
    )
    physical_corrections = (
        normalized_corrections * target_std + target_mean
    )

    nominal_states = nominal_transitions.state
    corrected_position = (
        nominal_states.quadrotor_state.p + physical_corrections[:, :3]
    )
    corrected_velocity = (
        nominal_states.quadrotor_state.v + physical_corrections[:, 3:6]
    )
    corrected_rotation = compose_orientation_residual(
        nominal_states.quadrotor_state.R,
        physical_corrections[:, 6:9],
    )
    corrected_acceleration = (
        corrected_velocity - env_states.quadrotor_state.v
    ) / sim_dt
    applied_body_rates = nominal_states.last_actions[
        :, effective_action_buffer_index, 1:
    ]
    corrected_quad_states = nominal_states.quadrotor_state.replace(
        p=corrected_position,
        R=corrected_rotation,
        v=corrected_velocity,
        acc=corrected_acceleration,
        omega=applied_body_rates,
    )
    corrected_states = nominal_states.replace(
        quadrotor_state=corrected_quad_states
    )
    raw_observations = jax.vmap(
        policy_sim_env._env._get_obs
    )(corrected_states)
    normalized_observations = (
        2.0
        * (raw_observations - policy_sim_env._obs_min)
        / (policy_sim_env._obs_max - policy_sim_env._obs_min)
        - 1.0
    )
    return corrected_states, normalized_observations, clipped_actions


def policy_rollout_loss(
    policy_params,
    joint_params,
    input_mean,
    input_std,
    target_mean,
    target_std,
    rollout_key,
):
    key_latents, key_reset, key_steps = jax.random.split(rollout_key, 3)
    rollout_latents = jax.random.normal(
        key_latents, (policy_num_envs, latent_dim)
    )
    reset_keys = jax.random.split(key_reset, policy_num_envs)
    initial_states, initial_observations = jax.vmap(
        lambda reset_key: policy_sim_env.reset(reset_key)
    )(reset_keys)
    scan_keys = jax.random.split(key_steps, policy_rollout_horizon)

    def rollout_step(carry, inputs):
        env_states, observations, active = carry
        step_index, step_key = inputs
        policy_inputs = jnp.concatenate(
            [observations, rollout_latents], axis=-1
        )
        actions = latent_policy_net.apply(policy_params, policy_inputs)
        environment_keys = jax.random.split(step_key, policy_num_envs)
        candidate_states, candidate_observations, clipped_actions = (
            corrected_policy_step(
                env_states,
                actions,
                rollout_latents,
                joint_params,
                input_mean,
                input_std,
                target_mean,
                target_std,
                environment_keys,
            )
        )
        remaining_steps = policy_rollout_horizon - step_index - 1
        unmasked_rewards, unmasked_position_error = (
            trajectory_tracking_reward(
                env_states,
                candidate_states,
                clipped_actions,
                remaining_steps,
            )
        )
        rewards = jnp.where(active, unmasked_rewards, 0.0)
        position_error = jnp.where(
            active, unmasked_position_error, 0.0
        )
        collision = jax.vmap(
            policy_sim_env._env._is_colliding
        )(candidate_states)
        next_active = active & ~collision
        next_states = jax.vmap(
            lambda is_active, candidate, previous: tree_select(
                is_active, candidate, previous
            )
        )(active, candidate_states, env_states)
        next_observations = jax.vmap(
            lambda is_active, candidate, previous: jax.lax.select(
                is_active, candidate, previous
            )
        )(active, candidate_observations, observations)
        outputs = (rewards, position_error, active)
        return (next_states, next_observations, next_active), outputs

    initial_active = jnp.ones(policy_num_envs, dtype=bool)
    _, (rewards, position_errors, active_steps) = jax.lax.scan(
        jax.checkpoint(rollout_step),
        (initial_states, initial_observations, initial_active),
        (jnp.arange(policy_rollout_horizon), scan_keys),
    )
    discounts = policy_discount ** jnp.arange(policy_rollout_horizon)
    discounted_returns = jnp.sum(
        discounts[:, None] * rewards, axis=0
    )
    loss = -discounted_returns.mean()
    auxiliary = (
        discounted_returns.mean(),
        rewards.sum() / jnp.maximum(active_steps.sum(), 1),
        position_errors.sum() / jnp.maximum(active_steps.sum(), 1),
        rollout_latents,
    )
    return loss, auxiliary


def tree_all_finite(tree):
    checks = [
        jnp.all(jnp.isfinite(leaf)) for leaf in jax.tree.leaves(tree)
    ]
    return jnp.all(jnp.stack(checks))


@jax.jit
def policy_optimizer_step(
    train_state,
    joint_params,
    input_mean,
    input_std,
    target_mean,
    target_std,
    rollout_key,
):
    (loss, auxiliary), gradients = jax.value_and_grad(
        policy_rollout_loss, has_aux=True
    )(
        train_state.params,
        joint_params,
        input_mean,
        input_std,
        target_mean,
        target_std,
        rollout_key,
    )
    gradient_norm = optax.global_norm(gradients)
    previous_params = train_state.params
    candidate_train_state = train_state.apply_gradients(grads=gradients)
    update_applied = (
        jnp.isfinite(loss)
        & tree_all_finite(gradients)
        & tree_all_finite(candidate_train_state.params)
        & tree_all_finite(candidate_train_state.opt_state)
    )
    train_state = jax.lax.cond(
        update_applied,
        lambda _: candidate_train_state,
        lambda _: train_state,
        operand=None,
    )
    update_norm = optax.global_norm(jax.tree.map(
        lambda updated, previous: updated - previous,
        train_state.params,
        previous_params,
    ))
    (
        mean_return, mean_reward, mean_position_error, rollout_latents
    ) = auxiliary
    return (
        train_state,
        loss,
        mean_return,
        mean_reward,
        mean_position_error,
        gradient_norm,
        update_norm,
        update_applied,
        rollout_latents,
    )


def run_policy_stage(
    train_state,
    joint_params,
    input_mean,
    input_std,
    target_mean,
    target_std,
    stage_key,
):
    stage_metrics = []
    sampled_prior_banks = []
    for policy_update in range(policy_updates_per_round):
        update_key = jax.random.fold_in(stage_key, policy_update)
        (
            train_state,
            loss,
            mean_return,
            mean_reward,
            mean_position_error,
            gradient_norm,
            update_norm,
            update_applied,
            rollout_latents,
        ) = policy_optimizer_step(
            train_state,
            joint_params,
            input_mean,
            input_std,
            target_mean,
            target_std,
            update_key,
        )
        stage_metrics.append(jnp.stack([
            loss, mean_return, mean_reward, mean_position_error,
            gradient_norm, update_norm, update_applied.astype(loss.dtype)
        ]))
        sampled_prior_banks.append(rollout_latents)
    metrics = jnp.stack(stage_metrics)
    next_stage_prior_bank = jnp.concatenate(sampled_prior_banks, axis=0)
    metrics.block_until_ready()
    next_stage_prior_bank.block_until_ready()
    return train_state, metrics, next_stage_prior_bank



## 10. Checkpointing

Every execution of the main training cell creates a new `run_YYYY-MM-DD_HH-MM-SS_microseconds` directory, so a new training cannot overwrite checkpoints from an earlier run. Round checkpoints are stored below that run directory. Each checkpoint contains both networks' optimizer states, the policy optimizer state, the current normalization statistics, the prior bank that must be reused by the next dynamics stage, and loop counters. Replay data remain in the in-memory accumulated buffer; rerunning from a checkpoint therefore requires separately persisting the deployment data if exact continuation is needed.



In [17]:
continual_checkpoint_base = os.path.abspath(os.path.join(
    LOTF_PATH,
    "..",
    "checkpoints",
    "continual_vnd",
    "traj_tracking_full",
))
os.makedirs(continual_checkpoint_base, exist_ok=True)


def create_continual_checkpoint_run():
    run_timestamp = datetime.now().strftime(
        "%Y-%m-%d_%H-%M-%S_%f"
    )
    checkpoint_root = os.path.join(
        continual_checkpoint_base, f"run_{run_timestamp}"
    )
    os.makedirs(checkpoint_root, exist_ok=False)
    return checkpoint_root


def save_continual_checkpoint(
    checkpoint_root,
    completed_rounds,
    joint_state,
    policy_state,
    input_mean,
    input_std,
    target_mean,
    target_std,
    fixed_prior_bank,
    global_dynamics_update,
):
    checkpoint_path = os.path.join(
        checkpoint_root, f"round_{completed_rounds:04d}"
    )
    checkpoint = {
        "joint_train_state": joint_state,
        "policy_train_state": policy_state,
        "input_mean": input_mean,
        "input_std": input_std,
        "residual_target_mean": target_mean,
        "residual_target_std": target_std,
        "next_dynamics_stage_prior_bank": fixed_prior_bank,
        "completed_rounds": jnp.asarray(completed_rounds),
        "global_dynamics_update": jnp.asarray(global_dynamics_update),
        "replay_num_groups": jnp.asarray(replay_buffer.num_groups),
    }
    checkpointer.save(checkpoint_path, checkpoint, force=True)
    return checkpoint_path



## 11. Run the alternating continual-learning pipeline

The ordering is intentional: Stage 1 uses the prior bank produced by the previous Stage 2; the first iteration uses an initial standard-normal bank. Stage 2 then samples the next bank while optimizing the policy. Simulated policy-improvement trajectories are not inserted into the deployment replay buffer.

Collection failure is nonfatal. Valid prefixes of terminated deployment rollouts contribute masked dynamics groups. If a round adds no group, Stage 1 trains from accumulated replay; if replay is still empty, only Stage 1 is skipped. Stage 2 always runs. During differentiable policy rollouts, the terminal transition and its penalty contribute to the objective, while rewards and state evolution after collision are masked.

For a short compilation/smoke test, temporarily set `num_continual_rounds=1`, `dynamics_updates_per_round=1`, `policy_num_envs` to a small value, and `policy_rollout_horizon` before running the configuration-dependent cells.



In [ ]:
continual_checkpoint_root = create_continual_checkpoint_run()
print("Checkpoint directory for this training run:")
print(f"  {continual_checkpoint_root}")

input_mean = startup_input_mean
input_std = startup_input_std
residual_target_mean = startup_residual_target_mean
residual_target_std = startup_residual_target_std

# The only prior sample created outside Stage 2: it is required by the first
# Stage 1 before any policy-improvement rollout has occurred.
fixed_prior_bank = jax.random.normal(
    experiment_key(100), (policy_num_envs, latent_dim)
)
global_dynamics_update = pretrained_vnd_completed_updates
if (
    initialize_vnd_from_pretrained_checkpoint
    and not update_normalization_from_continual_buffer
):
    print(
        "Using the pretrained VND normalization statistics. New replay "
        "groups will not replace them."
    )

training_history = {
    "dynamics_total": [],
    "dynamics_loss": [],
    "reconstruction_loss": [],
    "mmd_loss": [],
    "mmd_weight": [],
    "dynamics_gradient_norm": [],
    "dynamics_update_norm": [],
    "policy_loss": [],
    "policy_return": [],
    "policy_reward": [],
    "policy_position_error": [],
    "policy_gradient_norm": [],
    "policy_update_norm": [],
    "policy_update_applied_fraction": [],
    "replay_groups": [],
    "round_duration": [],
}

pipeline_start = time.perf_counter()
for continual_round in range(num_continual_rounds):
    round_start = time.perf_counter()
    round_key = jax.random.fold_in(experiment_key(200), continual_round)
    key_collection_base, key_prior_base, key_dynamics, key_policy = jax.random.split(
        round_key, 4
    )

    # Stages 1A/1B: retain valid prefixes from successful and failed
    # rollouts. Retry only when no rollout supplies a context plus one
    # valid target; failure to collect is nonfatal for the round.
    longest_valid_rollout = 0
    for collection_attempt in range(max_collection_attempts_per_round):
        key_collection = jax.random.fold_in(
            key_collection_base, collection_attempt
        )
        key_prior = jax.random.fold_in(key_prior_base, collection_attempt)
        rollout_states, rollout_actions, rollout_done = (
            collect_deployment_rollouts(
                policy_train_state.params,
                joint_train_state.params,
                input_mean,
                input_std,
                key_collection,
            )
        )
        valid_transition = (
            jnp.cumsum(rollout_done, axis=1) - rollout_done
        ) == 0
        longest_valid_rollout = max(
            longest_valid_rollout,
            int(jax.device_get(valid_transition.sum(axis=1).max())),
        )
        (
            new_contexts, new_dynamics_inputs, new_targets,
            new_target_masks,
        ) = (
            build_groups_from_collection(
                rollout_states, rollout_actions, rollout_done, key_prior
            )
        )
        if new_contexts.shape[0] > 0:
            break
        print(
            f"Collection attempt {collection_attempt + 1}/"
            f"{max_collection_attempts_per_round} produced no valid "
            f"{context_length}-step context plus target; retrying."
        )
    else:
        print(
            f"No new replay group after "
            f"{max_collection_attempts_per_round} collection batches. "
            f"The longest valid prefix had {longest_valid_rollout} "
            "steps. Continuing the round without new dynamics data."
        )

    new_group_count = new_contexts.shape[0]
    new_valid_target_count = int(jax.device_get(new_target_masks.sum()))
    if new_group_count > 0:
        replay_buffer.append(
            new_contexts, new_dynamics_inputs, new_targets,
            new_target_masks,
        )
    if update_normalization_from_continual_buffer and replay_buffer.num_groups:
        (
            input_mean,
            input_std,
            residual_target_mean,
            residual_target_std,
        ) = replay_buffer.statistics()

    # Stage 1C: train from accumulated replay, even when this round did
    # not add data. If replay is empty, skip only Stage 1C.
    dynamics_stage_ran = replay_buffer.num_groups > 0
    if dynamics_stage_ran:
        joint_train_state, dynamics_metrics = run_dynamics_stage(
            joint_train_state,
            replay_buffer,
            fixed_prior_bank,
            input_mean,
            input_std,
            residual_target_mean,
            residual_target_std,
            global_dynamics_update,
            key_dynamics,
        )
        global_dynamics_update += dynamics_updates_per_round
    else:
        dynamics_metrics = None
        print("Replay is empty; skipping only the dynamics stage.")

    # Stage 2: freeze VND, sample fresh rollout latents, and update the policy.
    policy_train_state, policy_metrics, fixed_prior_bank = run_policy_stage(
        policy_train_state,
        joint_train_state.params,
        input_mean,
        input_std,
        residual_target_mean,
        residual_target_std,
        key_policy,
    )

    host_dynamics_metrics = None
    if dynamics_stage_ran:
        host_dynamics_metrics = np.asarray(jax.device_get(dynamics_metrics))
        training_history["dynamics_total"].extend(
            host_dynamics_metrics[:, 0]
        )
        training_history["dynamics_loss"].extend(
            host_dynamics_metrics[:, 1]
        )
        training_history["reconstruction_loss"].extend(
            host_dynamics_metrics[:, 2]
        )
        training_history["mmd_loss"].extend(
            host_dynamics_metrics[:, 3]
        )
        training_history["mmd_weight"].extend(
            host_dynamics_metrics[:, 4]
        )
        training_history["dynamics_gradient_norm"].extend(
            host_dynamics_metrics[:, 5]
        )
        training_history["dynamics_update_norm"].extend(
            host_dynamics_metrics[:, 6]
        )
    host_policy_metrics = np.asarray(jax.device_get(policy_metrics))
    training_history["policy_loss"].append(host_policy_metrics[:, 0].mean())
    training_history["policy_return"].append(host_policy_metrics[:, 1].mean())
    training_history["policy_reward"].append(
        host_policy_metrics[:, 2].mean()
    )
    training_history["policy_position_error"].append(
        host_policy_metrics[:, 3].mean()
    )
    training_history["policy_gradient_norm"].append(
        host_policy_metrics[:, 4].mean()
    )
    training_history["policy_update_norm"].append(
        host_policy_metrics[:, 5].mean()
    )
    policy_update_applied_fraction = host_policy_metrics[:, 6].mean()
    training_history["policy_update_applied_fraction"].append(
        policy_update_applied_fraction
    )
    training_history["replay_groups"].append(replay_buffer.num_groups)
    round_duration = time.perf_counter() - round_start
    training_history["round_duration"].append(round_duration)

    if wandb_run is not None:
        diagnostic_raw_contexts = None
        if new_group_count > 0:
            diagnostic_raw_contexts = new_contexts[:min(256, new_group_count)]
        elif replay_buffer.num_groups > 0:
            diagnostic_sample_size = min(256, replay_buffer.num_groups)
            diagnostic_raw_contexts = replay_buffer.sample(
                jax.random.fold_in(round_key, 99), diagnostic_sample_size
            )[0]
        host_diagnostic_latents = None
        if diagnostic_raw_contexts is not None:
            diagnostic_contexts = normalize_state_actions(
                diagnostic_raw_contexts, input_mean, input_std
            )
            diagnostic_latents = trajectory_encoder.apply(
                joint_train_state.params["encoder"], diagnostic_contexts
            )
            host_diagnostic_latents = np.asarray(
                jax.device_get(diagnostic_latents)
            )
        encoder_parameter_norm = float(jax.device_get(
            optax.global_norm(joint_train_state.params["encoder"])
        ))
        residual_parameter_norm = float(jax.device_get(
            optax.global_norm(joint_train_state.params["residual"])
        ))
        decoder_parameter_norm = float(jax.device_get(
            optax.global_norm(joint_train_state.params["decoder"])
        ))
        policy_parameter_norm = float(jax.device_get(
            optax.global_norm(policy_train_state.params)
        ))
        round_log = {
            "continual/round": continual_round + 1,
            "policy/loss": host_policy_metrics[:, 0].mean(),
            "policy/return": host_policy_metrics[:, 1].mean(),
            "policy/mean_reward": host_policy_metrics[:, 2].mean(),
            "policy/position_error_m": (
                host_policy_metrics[:, 3].mean()
            ),
            "policy/gradient_norm": host_policy_metrics[:, 4].mean(),
            "policy/update_norm": host_policy_metrics[:, 5].mean(),
            "policy/update_applied_fraction": (
                policy_update_applied_fraction
            ),
            "dynamics_round/skipped": int(not dynamics_stage_ran),
            "replay/groups": replay_buffer.num_groups,
            "replay/new_groups": new_group_count,
            "replay/new_valid_targets": new_valid_target_count,
            "collection/attempts": collection_attempt + 1,
            "collection/longest_valid_steps": longest_valid_rollout,
            "parameters/encoder_norm": encoder_parameter_norm,
            "parameters/residual_norm": residual_parameter_norm,
            "parameters/decoder_norm": decoder_parameter_norm,
            "parameters/policy_norm": policy_parameter_norm,
            "timing/round_seconds": round_duration,
        }
        if dynamics_stage_ran:
            round_log.update({
                "dynamics_round/final_total_loss": host_dynamics_metrics[-1, 0],
                "dynamics_round/final_L_dyn": host_dynamics_metrics[-1, 1],
                "dynamics_round/final_L_rec": host_dynamics_metrics[-1, 2],
                "dynamics_round/final_L_mmd": host_dynamics_metrics[-1, 3],
                "dynamics_round/mean_L_dyn": host_dynamics_metrics[:, 1].mean(),
                "dynamics_round/mean_L_mmd": host_dynamics_metrics[:, 3].mean(),
            })
        if host_diagnostic_latents is not None:
            round_log.update({
                "encoder/latent_mean": host_diagnostic_latents.mean(),
                "encoder/latent_std": host_diagnostic_latents.std(),
                "encoder/latent_abs_mean": np.abs(
                    host_diagnostic_latents
                ).mean(),
                "encoder/latent_max_abs": np.abs(
                    host_diagnostic_latents
                ).max(),
            })
            latent_dimension_means = host_diagnostic_latents.mean(axis=0)
            latent_dimension_stds = host_diagnostic_latents.std(axis=0)
            for latent_index in range(latent_dim):
                round_log[f"encoder/latent_mean_{latent_index:02d}"] = (
                    latent_dimension_means[latent_index]
                )
                round_log[f"encoder/latent_std_{latent_index:02d}"] = (
                    latent_dimension_stds[latent_index]
                )
            if (
                wandb_histogram_every_rounds
                and (continual_round + 1) % wandb_histogram_every_rounds == 0
            ):
                round_log["encoder/latent_histogram"] = wandb.Histogram(
                    host_diagnostic_latents
                )
        wandb_run.log(round_log)

    if checkpoint_every_rounds and (
        (continual_round + 1) % checkpoint_every_rounds == 0
        or continual_round + 1 == num_continual_rounds
    ):
        saved_path = save_continual_checkpoint(
            continual_checkpoint_root,
            continual_round + 1,
            joint_train_state,
            policy_train_state,
            input_mean,
            input_std,
            residual_target_mean,
            residual_target_std,
            fixed_prior_bank,
            global_dynamics_update,
        )
    else:
        saved_path = None

    if dynamics_stage_ran:
        dynamics_status = (
            f"L_dyn {host_dynamics_metrics[-1, 1]:.4e} | "
            f"L_mmd {host_dynamics_metrics[-1, 3]:.4e} | "
            f"lambda_mmd {host_dynamics_metrics[-1, 4]:.2e}"
        )
    else:
        dynamics_status = "dynamics skipped (empty replay)"
    if policy_update_applied_fraction < 1.0:
        print(
            "WARNING: skipped a policy update because its loss, "
            "gradients, parameters, or optimizer state were non-finite."
        )
    print(
        f"Round {continual_round + 1:4d}/{num_continual_rounds} | "
        f"buffer {replay_buffer.num_groups:7d} groups | "
        f"new {new_group_count} groups/"
        f"{new_valid_target_count} targets | {dynamics_status} | "
        f"policy return {host_policy_metrics[:, 1].mean():.4f} | "
        f"mean reward {host_policy_metrics[:, 2].mean():.4f} | "
        f"position error {host_policy_metrics[:, 3].mean():.4f} m | "
        f"{training_history['round_duration'][-1]:.1f} s"
    )
    if saved_path is not None:
        print(f"  checkpoint: {saved_path}")

total_pipeline_seconds = time.perf_counter() - pipeline_start
print(
    f"Completed {num_continual_rounds} rounds in "
    f"{total_pipeline_seconds:.1f} s"
)
if wandb_run is not None:
    wandb_run.summary["completed_rounds"] = num_continual_rounds
    wandb_run.summary["total_pipeline_seconds"] = total_pipeline_seconds
    wandb_run.finish()



Checkpoint directory for this training run:
  /home/dimitria/PhD_codes/learning_on_the_fly/checkpoints/continual_vnd/traj_tracking_full/run_2026-08-28_13-50-39_114047
Using the pretrained VND normalization statistics. New replay groups will not replace them.
Round    1/500 | buffer     256 groups | new 256 groups/20408 targets | L_dyn 6.9616e-06 | L_mmd 5.6762e-02 | lambda_mmd 1.00e-04 | policy return -0.4708 | mean reward -0.0042 | position error 0.3943 m | 36.8 s
Round    2/500 | buffer     512 groups | new 256 groups/19205 targets | L_dyn 3.9388e-05 | L_mmd 5.3452e-02 | lambda_mmd 1.00e-04 | policy return -1.4552 | mean reward -0.0147 | position error 0.8098 m | 12.3 s
Round    3/500 | buffer     768 groups | new 256 groups/23424 targets | L_dyn 5.7520e-05 | L_mmd 4.3840e-02 | lambda_mmd 1.00e-04 | policy return -0.5310 | mean reward -0.0044 | position error 0.4527 m | 4.7 s
Round    4/500 | buffer    1024 groups | new 256 groups/25801 targets | L_dyn 7.5096e-05 | L_mmd 4.9964e-02 |

## 12. Training curves and sampled-latent sanity check

These plots report optimizer objectives, not held-out dynamics validation. The final print checks only the marginal mean and standard deviation of the sampled Stage-2 prior bank; MMD alignment of encoded latents should be evaluated separately on encoded buffer contexts.



In [ ]:
dynamics_update_axis = np.arange(len(training_history["dynamics_loss"]))
round_axis = np.arange(1, len(training_history["policy_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(dynamics_update_axis, training_history["dynamics_loss"])
axes[0, 0].set(title="VND dynamics loss", xlabel="Dynamics update", ylabel="L_dyn")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(dynamics_update_axis, training_history["mmd_loss"], label="L_mmd")
axes[0, 1].plot(dynamics_update_axis, training_history["mmd_weight"], label="lambda_mmd")
axes[0, 1].set(title="Prior alignment", xlabel="Dynamics update")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(round_axis, training_history["policy_return"])
axes[1, 0].set(title="BPTT policy return", xlabel="Continual round", ylabel="Mean return")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(round_axis, training_history["policy_position_error"])
axes[1, 1].set(
    title="BPTT mean trajectory-tracking position error",
    xlabel="Continual round",
    ylabel="Position error [m]",
)
axes[1, 1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

final_prior_mean = fixed_prior_bank.mean(axis=0)
final_prior_std = fixed_prior_bank.std(axis=0)
print("Final Stage-2 sampled prior bank:")
print(f"  mean per latent dimension: {final_prior_mean}")
print(f"  std per latent dimension:  {final_prior_std}")
print(f"  aggregate mean: {final_prior_mean.mean():.4f}")
print(f"  aggregate std:  {final_prior_std.mean():.4f}")
